# 阶段 0：随机 20 个 Study 的 DICOM 解码审计

目标：随机抽取至少 20 个训练集 study，定位其 series 和 DICOM 切片，记录 Transfer Syntax，并实际读取像素数组。无法定位或解码的样本都会写入日志，不会被静默忽略。

运行前：在右侧 Add Input 中添加 RSNA Knee Abnormality Detection 比赛数据。本 Notebook 不使用网络。

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import pydicom
from pydicom.uid import UID

SEED = 20260824
N_STUDIES = 20
MAX_SERIES_PER_STUDY = 5
INPUT_ROOT = Path('/kaggle/input')
WORKING_ROOT = Path('/kaggle/working')
rng = np.random.default_rng(SEED)

print('pydicom:', pydicom.__version__)
print('image handlers:', [h.__name__ for h in pydicom.config.pixel_data_handlers])

In [ ]:
# 优先使用参考 Notebook 中的比赛数据路径；同时兼容其它 Kaggle 挂载名称。
known_train_csv = INPUT_ROOT / 'competitions' / 'rsna-knee-abnormality-detection' / 'train.csv'
candidates = ([known_train_csv] if known_train_csv.exists() else []) + list(INPUT_ROOT.glob('*/train.csv'))
if not candidates:
    raise FileNotFoundError('未找到 /kaggle/input/*/train.csv。请先在 Add Input 中添加比赛数据。')
DATA_ROOT = candidates[0].parent
train = pd.read_csv(DATA_ROOT / 'train.csv')
train_series = pd.read_csv(DATA_ROOT / 'train_series.csv')
print('DATA_ROOT =', DATA_ROOT)
print('train:', train.shape, '| train_series:', train_series.shape)
display(train.head(3))
display(train_series.head(3))

In [ ]:
def required_column(frame, options, label):
    for col in options:
        if col in frame.columns:
            return col
    raise KeyError(f'找不到 {label}。可用列：{frame.columns.tolist()}')

STUDY_COL = required_column(train_series, ['StudyInstanceUID', 'study_id', 'StudyUID'], 'study UID 列')
SERIES_COL = required_column(train_series, ['SeriesInstanceUID', 'series_id', 'SeriesUID'], 'series UID 列')
PATH_COL = next((c for c in ['path', 'Path', 'series_path', 'SeriesPath', 'dicom_path'] if c in train_series.columns), None)
study_ids = train_series[STUDY_COL].dropna().drop_duplicates().to_numpy()
if len(study_ids) < N_STUDIES:
    raise ValueError(f'只找到 {len(study_ids)} 个 study。')
sampled_studies = rng.choice(study_ids, N_STUDIES, replace=False).tolist()
print(f'抽样 study 数：{len(sampled_studies)}；路径列：{PATH_COL}')

In [ ]:
# 只查询已抽样 study 的候选目录，避免递归扫描整个数百 GB 数据集。
DICOM_SUFFIXES = {'.dcm', '.dicom', ''}
IMAGE_ROOTS = [p for p in DATA_ROOT.iterdir() if p.is_dir()]

def files_in(folder):
    if not folder or not folder.is_dir():
        return []
    return sorted(p for p in folder.iterdir() if p.is_file() and p.suffix.lower() in DICOM_SUFFIXES)

def locate_files(row):
    study_uid, series_uid = str(row[STUDY_COL]), str(row[SERIES_COL])
    folders = []
    if PATH_COL and pd.notna(row[PATH_COL]):
        raw = Path(str(row[PATH_COL]))
        folders.extend([raw, DATA_ROOT / raw, INPUT_ROOT / raw])
    for root in IMAGE_ROOTS:
        folders.extend([root / study_uid / series_uid, root / study_uid, root / series_uid])
    seen = set()
    for folder in folders:
        key = str(folder)
        if key in seen:
            continue
        seen.add(key)
        files = files_in(folder)
        if files:
            return files, folder
    return [], None

def decode_one(path):
    header = pydicom.dcmread(path, stop_before_pixels=True, force=False)
    transfer_syntax = str(getattr(header.file_meta, 'TransferSyntaxUID', 'MISSING'))
    ds = pydicom.dcmread(path, force=False)
    pixels = ds.pixel_array  # 访问 pixel_array 才算实际验证解码
    uid = UID(transfer_syntax) if transfer_syntax != 'MISSING' else None
    return {
        'transfer_syntax_uid': transfer_syntax,
        'transfer_syntax_name': uid.name if uid else 'MISSING',
        'is_compressed': uid.is_compressed if uid else None,
        'pixel_shape': tuple(pixels.shape),
        'pixel_dtype': str(pixels.dtype),
        'modality': getattr(ds, 'Modality', None),
    }

In [ ]:
audit_rows, failure_rows = [], []
for study_uid in sampled_studies:
    series_rows = train_series.loc[train_series[STUDY_COL] == study_uid].head(MAX_SERIES_PER_STUDY)
    for _, row in series_rows.iterrows():
        series_uid = str(row[SERIES_COL])
        files, folder = locate_files(row)
        if not files:
            failure_rows.append({'study_uid': study_uid, 'series_uid': series_uid, 'stage': 'file_lookup', 'error': 'No DICOM files found'})
            continue
        path = files[0]
        try:
            audit_rows.append({'study_uid': study_uid, 'series_uid': series_uid, 'series_folder': str(folder), 'dicom_file': str(path), 'series_file_count': len(files), 'decode_ok': True, **decode_one(path)})
        except Exception as exc:
            failure_rows.append({'study_uid': study_uid, 'series_uid': series_uid, 'stage': 'pixel_decode', 'dicom_file': str(path), 'error_type': type(exc).__name__, 'error': repr(exc)})

audit, failures = pd.DataFrame(audit_rows), pd.DataFrame(failure_rows)
print(f'抽样 study：{len(sampled_studies)}；成功解码 series：{len(audit)}；失败记录：{len(failures)}')
display(audit.head())
display(failures.head())

In [ ]:
if audit.empty:
    raise RuntimeError('没有成功解码的 DICOM。请查看 failures，并核对数据挂载路径与可用解码器。')
summary = (audit.groupby(['transfer_syntax_uid', 'transfer_syntax_name', 'is_compressed'], dropna=False)
           .agg(decoded_series=('series_uid', 'nunique'), decoded_studies=('study_uid', 'nunique'), example_file=('dicom_file', 'first'))
           .reset_index().sort_values(['decoded_studies', 'decoded_series'], ascending=False))
display(summary)
missing_studies = sorted(set(sampled_studies) - set(audit['study_uid']))
print('未成功解码任一 series 的 study 数：', len(missing_studies))

audit.to_csv(WORKING_ROOT / 'dicom_decode_audit.csv', index=False)
failures.to_csv(WORKING_ROOT / 'dicom_decode_failures.csv', index=False)
summary.to_csv(WORKING_ROOT / 'dicom_transfer_syntax_summary.csv', index=False)
print('已保存三份审计 CSV 到 /kaggle/working。')

## 阶段 0 验收

- 至少 20 个随机 study 中，每个被抽取的 series 都有成功或失败记录。
- 传输语法汇总表展示每种实际出现的传输语法及其解码结果。
- 出现失败时保留失败 CSV；不要静默跳过。后续可按错误类型确认是否需要 JPEG-LS、JPEG 2000 或 RLE 解码器。